## Vanilla Mamba — Inference Notebook

Load the trained Mamba checkpoint and generate text from prompts.

## 1 · Imports & Config

In [1]:
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer
from model import MambaModel

# ── Model hyper-parameters (must match training) ────────────────────
VOCAB_SIZE = 8192
D_MODEL    = 256
N_LAYERS   = 6
D_STATE    = 16
D_CONV     = 4
EXPAND     = 2

# ── Special tokens ─────────────────────────────────────────────────
BOS_ID = 2
EOS_ID = 3

# ── Paths ──────────────────────────────────────────────────────────
CHECKPOINT_PATH = "best.pt"
TOKENIZER_PATH  = "tokenizer.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cpu


## 2 · Load Tokenizer

In [2]:
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
print(f"Tokenizer loaded  ·  vocab size = {tokenizer.get_vocab_size()}")

Tokenizer loaded  ·  vocab size = 8192


## 3 · Load Model & Checkpoint

In [3]:
model = MambaModel(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_layers=N_LAYERS,
    d_state=D_STATE,
    d_conv=D_CONV,
    expand=EXPAND,
).to(DEVICE)

state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(state_dict)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded  ·  {n_params / 1e6:.2f}M parameters")

Model loaded  ·  6.73M parameters


## 4 · Generation Utilities

In [4]:
@torch.no_grad()
def generate(
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float = 0.8,
    top_k: int = 50,
    top_p: float = 0.95,
):
    """
    Auto-regressive generation with temperature, top-k, and nucleus (top-p) sampling.
    """
    encoded = tokenizer.encode(prompt)
    input_ids = torch.tensor(
        [BOS_ID] + encoded.ids, dtype=torch.long
    ).unsqueeze(0).to(DEVICE)

    for _ in range(max_new_tokens):
        logits = model(input_ids)
        next_logits = logits[:, -1, :] / temperature

        # ── Top-k filtering ────────────────────────────────────────
        if top_k > 0:
            top_k_vals, _ = torch.topk(next_logits, top_k)
            threshold = top_k_vals[:, -1].unsqueeze(-1)
            next_logits = next_logits.masked_fill(next_logits < threshold, -float('inf'))

        # ── Nucleus (top-p) filtering ──────────────────────────────
        if top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(next_logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            remove_mask = cumulative_probs - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[remove_mask] = -float('inf')
            next_logits = sorted_logits.scatter(1, sorted_idx, sorted_logits)

        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=-1)

        if next_token.item() == EOS_ID:
            break

    return tokenizer.decode(input_ids[0].tolist())


def show(prompt, **kwargs):
    """Pretty-print a generation."""
    text = generate(prompt, **kwargs)
    print(f"\n{'─' * 60}")
    print(f"Prompt │ {prompt}")
    print(f"{'─' * 60}")
    print(text)
    print(f"{'─' * 60}\n")

## 5 · Generate Stories 📖

In [5]:
show("Once upon a time")


────────────────────────────────────────────────────────────
Prompt │ Once upon a time
────────────────────────────────────────────────────────────
Once upon a time, there was a little boy named Timmy. Timmy loved to play with her toys and play with her toys in the park.
────────────────────────────────────────────────────────────



In [6]:
show("The little cat")


────────────────────────────────────────────────────────────
Prompt │ The little cat
────────────────────────────────────────────────────────────
The little cat named Max were playing with him.

"Look, Lily, a little girl named Lucy came into the water. They wanted to play with the animals in the park. It was a small, red thing in the lake. It was a big slide. He ran and tried to take a walk with the dog.

"Look, Ben, please?" Sam asked.

"You can't. You need to see this food!"

Sally's mom told her, "Can I play with the dog and have fun."

But Lily was so happy.

"Hey, you should not try to give it to you. You are not a bunny. He says, "Lily, the man was angry. He wanted to play outside and play with his toys. They are good for the toy. From that day on, Tom and her friend, the wind on the table. He had a lot of fun and they went on a walk. He
────────────────────────────────────────────────────────────



In [7]:
show("There was a boy named")


────────────────────────────────────────────────────────────
Prompt │ There was a boy named
────────────────────────────────────────────────────────────
There was a boy named Max. Timmy was very proud of herself for being brave and yard.
────────────────────────────────────────────────────────────



## 6 · Experiment with Sampling Parameters

Try different temperatures, top-k, and top-p values to see how generation changes.

In [8]:
prompt = "Once upon a time"

print("=" * 60)
print("GREEDY  (temperature → 0)")
print("=" * 60)
show(prompt, temperature=0.1, top_k=1)

print("=" * 60)
print("WARM  (temperature = 0.8, top_k = 50)")
print("=" * 60)
show(prompt, temperature=0.8, top_k=50)

print("=" * 60)
print("CREATIVE  (temperature = 1.2, top_p = 0.9)")
print("=" * 60)
show(prompt, temperature=1.2, top_k=0, top_p=0.9)

GREEDY  (temperature → 0)

────────────────────────────────────────────────────────────
Prompt │ Once upon a time
────────────────────────────────────────────────────────────
Once upon a time, there was a little girl named Lily. She loved to play outside in the park. They saw a big tree with a big smile on her face. She was very happy and thanked the bird. They were happy. They had a lot of fun.

"I'm sorry, Lily. I will be a good friend. I'm sorry. I will be a good friend. I'm sorry. I will be a good friend. I'm sorry. I will be a good friend. I'm sorry. I will be a good friend. I'm sorry. I will be a good friend. I'm sorry. I will be a good friend. I'm sorry. I will be a good friend. I'm sorry. I will be a good friend. I'm sorry. I will be a good friend. I'm sorry. I will be a good friend. I'm sorry. I will be a good friend. I'm sorry. I will be a good friend. I
────────────────────────────────────────────────────────────

WARM  (temperature = 0.8, top_k = 50)

──────────────────────